# LoCoMo QA Speaker Mention Check

`dataset/locomo10.json`의 모든 sample에 대해 `qa[*].question` 안에 해당 sample의 두 화자가 직접 명시되는지 집계하는 노트북입니다.

집계 기준은 아래와 같습니다.
- `speaker1 = conversation.speaker_a`
- `speaker2 = conversation.speaker_b`
- exact match만 인정합니다.
- `Caroline's` 같은 소유격은 `Caroline`이 직접 등장하므로 명시로 봅니다.
- 별칭(`Mel`)이나 오탈자/변형(`Johns's`)은 명시로 보지 않습니다.
- 각 질문은 아래 네 가지 중 하나로만 분류합니다.
  - `speaker1 only`
  - `speaker2 only`
  - `both`
  - `others`


In [1]:
from __future__ import annotations

import json
import re
from collections import Counter
from pathlib import Path

from IPython.display import HTML, Markdown, display

try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None


if pd is not None:
    pd.set_option("display.max_colwidth", None)
    pd.set_option("display.max_columns", None)


def rows_to_html(rows: list[dict], columns: list[str] | None = None) -> HTML:
    if not rows:
        return HTML("<p><i>No rows</i></p>")

    if columns is None:
        columns = list(rows[0].keys())

    header_html = "".join(f"<th>{column}</th>" for column in columns)
    body_html = []
    for row in rows:
        cells = "".join(f"<td>{row.get(column, '')}</td>" for column in columns)
        body_html.append(f"<tr>{cells}</tr>")

    table_html = (
        "<table border='1' style='border-collapse:collapse'>"
        f"<thead><tr>{header_html}</tr></thead>"
        f"<tbody>{''.join(body_html)}</tbody>"
        "</table>"
    )
    return HTML(table_html)


def display_table(rows: list[dict], columns: list[str] | None = None):
    if pd is not None:
        df = pd.DataFrame(rows)
        if columns is not None:
            df = df[columns]
        display(df)
        return df

    display(rows_to_html(rows, columns=columns))
    return rows


In [2]:
DATA_PATH = Path("dataset/locomo10.json")

with DATA_PATH.open("r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} samples from {DATA_PATH}")

Loaded 10 samples from dataset/locomo10.json


In [3]:
def has_exact_name_mention(question: str, speaker_name: str) -> bool:
    pattern = re.compile(rf"\b{re.escape(speaker_name)}\b")
    return pattern.search(question) is not None


def classify_question(question: str, speaker1: str, speaker2: str) -> str:
    has_speaker1 = has_exact_name_mention(question, speaker1)
    has_speaker2 = has_exact_name_mention(question, speaker2)

    if has_speaker1 and has_speaker2:
        return "both"
    if has_speaker1:
        return "speaker1 only"
    if has_speaker2:
        return "speaker2 only"
    return "others"


rows = []

for sample in data:
    conversation = sample["conversation"]
    speaker1 = conversation["speaker_a"]
    speaker2 = conversation["speaker_b"]

    for qa_idx, qa in enumerate(sample.get("qa", []), start=1):
        question = qa["question"]
        label = classify_question(question, speaker1, speaker2)
        rows.append(
            {
                "sample_id": sample["sample_id"],
                "qa_index": qa_idx,
                "speaker1": speaker1,
                "speaker2": speaker2,
                "question": question,
                "label": label,
            }
        )

label_counter = Counter(row["label"] for row in rows)
summary_rows = [
    {"bucket": "speaker1 명시", "count": label_counter["speaker1 only"]},
    {"bucket": "speaker2 명시", "count": label_counter["speaker2 only"]},
    {"bucket": "speaker1, speaker2 명시", "count": label_counter["both"]},
    {"bucket": "others", "count": label_counter["others"]},
]

others_rows = [row for row in rows if row["label"] == "others"]


In [4]:
display(Markdown("## Summary"))
for row in summary_rows:
    print(f"{row['bucket']}: {row['count']}개")

display_table(summary_rows, columns=["bucket", "count"])

## Summary

speaker1 명시: 928개
speaker2 명시: 885개
speaker1, speaker2 명시: 151개
others: 22개


,bucket,count
0,speaker1 명시,928
1,speaker2 명시,885
2,"speaker1, speaker2 명시",151
3,others,22


,bucket,count
0,speaker1 명시,928
1,speaker2 명시,885
2,"speaker1, speaker2 명시",151
3,others,22


In [5]:
display(Markdown("## Others"))
print(f"others 질문 수: {len(others_rows)}개")

others_df = display_table(
    others_rows,
    columns=["sample_id", "qa_index", "speaker1", "speaker2", "question"],
)

others_df

## Others

others 질문 수: 22개


,sample_id,qa_index,speaker1,speaker2,question
0,conv-26,83,Caroline,Melanie,What did the charity race raise awareness for?
1,conv-26,91,Caroline,Melanie,How long have Mel and her husband been married?
2,conv-26,99,Caroline,Melanie,What was discussed in the LGBTQ+ counseling workshop?
3,conv-26,110,Caroline,Melanie,What did Mel and her kids make during the pottery workshop?
4,conv-26,111,Caroline,Melanie,What kind of pot did Mel and her kids make with clay?
5,conv-26,112,Caroline,Melanie,What creative project do Mel and her kids do together besides pottery?
6,conv-26,113,Caroline,Melanie,What did Mel and her kids paint in their latest project in July 2023?
7,conv-26,126,Caroline,Melanie,Where did Oliver hide his bone once?
8,conv-26,141,Caroline,Melanie,What did the posters at the poetry reading say?
9,conv-26,180,Caroline,Melanie,Where did Oscar hide his bone once?


,sample_id,qa_index,speaker1,speaker2,question
0,conv-26,83,Caroline,Melanie,What did the charity race raise awareness for?
1,conv-26,91,Caroline,Melanie,How long have Mel and her husband been married?
2,conv-26,99,Caroline,Melanie,What was discussed in the LGBTQ+ counseling workshop?
3,conv-26,110,Caroline,Melanie,What did Mel and her kids make during the pottery workshop?
4,conv-26,111,Caroline,Melanie,What kind of pot did Mel and her kids make with clay?
5,conv-26,112,Caroline,Melanie,What creative project do Mel and her kids do together besides pottery?
6,conv-26,113,Caroline,Melanie,What did Mel and her kids paint in their latest project in July 2023?
7,conv-26,126,Caroline,Melanie,Where did Oliver hide his bone once?
8,conv-26,141,Caroline,Melanie,What did the posters at the poetry reading say?
9,conv-26,180,Caroline,Melanie,Where did Oscar hide his bone once?
